# Chapter 2 – Worked examples

This notebook runs the examples of [Chapter 2](https://katra.ballardini.com.ar/power-grid-topology-optimization-course/devdev/part-1-math/ch02-linear-algebra/) live. The code lives in the modules next to it
(`laplacian.py`, `lab_network.py`, `resistor_lab.py`); the cells only call them, so a change in the module is a
change here too.

Run it from this folder (the kernel's working directory must be `src/`), or open it in Jupyter Lab from the
repository.

In [ ]:
# JupyterLite only: install the libraries this notebook and its chapter modules use.
%pip install -q matplotlib numpy scipy

In [ ]:
import laplacian
import numpy as np
import resistor_lab

np.set_printoptions(precision=4, suppress=True, linewidth=119)

## Four nodes, four branches

Branch $k$ goes from node $f_k$ to node $t_k$ with conductance $g_k$. The incidence matrix has $+1$ at the from
node and $-1$ at the to node of each row, and $G = A^\top \operatorname{diag}(g) A$ is the weighted Laplacian.

In [ ]:
a = laplacian.incidence(laplacian.fr, laplacian.to, 4)
gmat = laplacian.laplacian(a, laplacian.g)
print('A =\n', a)
print('G =\n', gmat)
print('row sums of A:', a.sum(axis=1), '  G 1 =', gmat @ np.ones(4))

Every row of $A$ sums to zero, so $G\mathbf{1} = 0$: one zero eigenvalue, with the constant vector as
eigenvector. The network is connected, so there is exactly one.

In [ ]:
w, vecs = np.linalg.eigh(gmat)
print('eigenvalues', w)
print('eigenvector of the zero eigenvalue', vecs[:, 0])

## Grounding a node makes $G$ invertible

Inject 1 A at node 0 and withdraw it at node 3. With node 0 at 0 V (its row and column dropped) the reduced
system has a unique solution; the branch currents $g \odot (Av)$ satisfy Kirchhoff's current law at every node,
the grounded one included.

In [ ]:
i = np.array([1.0, 0.0, 0.0, -1.0])
v = laplacian.nodal_voltages(gmat, i)
cur = resistor_lab.branch_currents(a, laplacian.g, v)
print('v =', v)
print('branch currents =', cur)
print('A^T (g * A v) - i =', a.T @ cur - i)

## The lab network

The six-node network of the lab, built the same way. `resistor_lab.checks` reports symmetry, row sums and the
number of zero eigenvalues.

In [ ]:
a6, g6 = resistor_lab.build(resistor_lab.FR, resistor_lab.TO, resistor_lab.G, resistor_lab.N_NODES)
print(resistor_lab.checks(g6))

Try it yourself: change one conductance in `laplacian.g` (a copy) and watch which entries of $G$ move; then set
one to zero and see whether the number of zero eigenvalues changes.

In [ ]:
g_try = laplacian.g.copy()
g_try[3] = 0.0  # branch 2 -> 3 removed: node 3 is isolated
print(np.linalg.eigvalsh(laplacian.laplacian(a, g_try)))